In [ ]:
# calculate metrics by entity types
import json
from collections import Counter
import pandas as pd

# define question entity map
QUESTION_ENTITY_MAP = {
    "What is the name of the host protein/RBP/host factor interacting with the virus?": "Host Protein",
    "What experimental methods were used to detect the virus-host interaction?": "Experimental Method",
    "What is the infection time for the experiment?": "Infection Time",
    "What is the name of the virus whose protein interacts with host factors/proteins?": "Virus_PPI",
    "What is the name of the virus whose RNA interacts with host factors/proteins?": "Virus_RPI",
    "What type of cell was infected by the virus in this experiment?": "Cell Type",
    "What function does the host protein have on the virus ?": "Host Protein Function",
    "What is the name of the virus protein interacting with the host protein/RBP/host factor?": "Virus Protein",
    "What is the host species that virus infected?": "Host Species",
    "What is the strain/subtype of the virus studied?": "Virus Strain",
    "Which RNA structures within the viral genome are preferentially bound by host proteins?": "RNA Structure Preference",
    "Where is the binding site/region located on the Virus?": "Binding Site",
    "What tissue/organ does the infected cell originate from?": "Cell Origin",
    "What is the name of the table that includes the interaction between viral RNA and host protein?": "Table Name",
    "What function does the virus protein have on the host?": "Virus Protein Function",
}

def normalize(text):
    return text.strip().rstrip("?").lower()
normalized_question_map = {normalize(q): v for q, v in QUESTION_ENTITY_MAP.items()}

# BIO label extraction function
def get_entities(seq, id2label, markup='bios'):
    chunks = []
    chunk = [-1, -1, -1]
    for i, tag in enumerate(seq):
        label = id2label[tag] if isinstance(tag, int) else tag
        if label.startswith("B-"):
            if chunk[2] != -1:
                chunks.append(tuple(chunk))
            chunk = [label[2:], i, i]
        elif label.startswith("I-") and chunk[0] == label[2:]:
            chunk[2] = i
        else:
            if chunk[2] != -1:
                chunks.append(tuple(chunk))
            chunk = [-1, -1, -1]
    if chunk[2] != -1:
        chunks.append(tuple(chunk))
    return chunks

# BIO evaluator
class SeqEntityScore:
    def __init__(self, id2label, markup='bios'):
        self.id2label = id2label
        self.markup = markup
        self.reset()

    def reset(self):
        self.origins, self.founds, self.rights = [], [], []

    def compute(self, origin, found, right):
        recall = 0 if origin == 0 else (right / origin)
        precision = 0 if found == 0 else (right / found)
        f1 = 0. if recall + precision == 0 else (2 * precision * recall) / (precision + recall)
        return recall, precision, f1

    def result(self):
        r, p, f1 = self.compute(len(self.origins), len(self.founds), len(self.rights))
        return {"precision": round(p,4), "recall": round(r,4), "f1": round(f1,4)}

    def update(self, label_paths, pred_paths):
        for label_path, pre_path in zip(label_paths, pred_paths):
            gold = get_entities(label_path, self.id2label, self.markup)
            pred = get_entities(pre_path, self.id2label, self.markup)
            self.origins.extend(gold)
            self.founds.extend(pred)
            self.rights.extend([p for p in pred if p in gold])

# BIO label generator
def make_bio_labels(context, spans, label="ANSWER"):
    tokens = context.split()
    token_starts = []
    idx = 0
    for token in tokens:
        idx = context.find(token, idx)
        token_starts.append(idx)
        idx += len(token)
    labels = ["O"] * len(tokens)
    for start_char, end_char in spans:
        for i, token_start in enumerate(token_starts):
            token_end = token_start + len(tokens[i])
            if token_end <= start_char: continue
            if token_start >= end_char: break
            labels[i] = "B-" + label if labels[i] == "O" else "I-" + label
    return tokens, labels

# load data
with open("/content/drive/MyDrive/数据/Test/Data_test_fixed.json", "r", encoding="utf-8") as f:
    ground_truth = [json.loads(line) for line in f]
with open("/content/drive/MyDrive/数据/100%_data_train_GPT_4.1_results/predict_predictions.json", "r", encoding="utf-8") as f:
    predictions = json.load(f)

id2label = {0: "O", 1: "B-ANSWER", 2: "I-ANSWER"}

# every entity type with independent evaluator
evaluators = {etype: SeqEntityScore(id2label) for etype in set(QUESTION_ENTITY_MAP.values())}

for item in ground_truth:
    norm_q = normalize(item["question"])
    entity_type = normalized_question_map.get(norm_q)
    if entity_type is None:
        continue
    qid = item["id"]
    context = item["context"].lower()
    true_text = item["answers"]["text"][0].strip().lower()
    true_start = item["answers"]["answer_start"][0]
    true_end = true_start + len(true_text)
    _, true_labels = make_bio_labels(context, [(true_start, true_end)])

    if qid in predictions:
        pred_text = predictions[qid].strip().lower()
        pred_start = context.find(pred_text)
        if pred_start != -1:
            pred_end = pred_start + len(pred_text)
            _, pred_labels = make_bio_labels(context, [(pred_start, pred_end)])
        else:
            _, pred_labels = make_bio_labels(context, [])
    else:
        _, pred_labels = make_bio_labels(context, [])

    evaluators[entity_type].update([true_labels], [pred_labels])

# Metrics results by labels
results = []
for etype, evaluator in evaluators.items():
    metrics = evaluator.result()
    results.append({
        "Entity Type": etype,
        "Precision": metrics["precision"],
        "Recall": metrics["recall"],
        "F1": metrics["f1"]
    })

# DataFrame
df_results = pd.DataFrame(results).sort_values(by="F1", ascending=False).reset_index(drop=True)
df_results

,Entity Type,Precision,Recall,F1
0,Cell Origin,1.0000,1.0000,1.0000
1,Virus_RPI,1.0000,1.0000,1.0000
2,Infection Time,1.0000,1.0000,1.0000
3,RNA Structure Preference,1.0000,1.0000,1.0000
4,Table Name,1.0000,1.0000,1.0000
5,Virus_PPI,0.9783,1.0000,0.9890
6,Virus Protein,0.9596,0.9406,0.9500
7,Binding Site,0.8605,0.9867,0.9193
8,Host Species,0.8462,1.0000,0.9167
9,Cell Type,0.9859,0.8046,0.8861
